# Thai Constitution Post-Processor v2
---
For Fixxing 3 feedback

**Bug 1 — Bunch ไม่คม**

**Bug 2 — ไม่รู้จัก "ส่วนที่"**

**Bug 3 — ประกาศแก้ไข ref กลับไม่ได้**

In [7]:
import re
import json
import csv
import argparse
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

## 1. TEXT CLEANING
---

In [8]:
_HEADER_PATTERNS = [
    re.compile(r'วัน(?:ที่|พุธที่|จันทร์ที่|อังคารที่|พฤหัสที่|ศุกร์ที่|เสาร์ที่|อาทิตย์ที่)?\s*[\d๐-๙]+\s*\S+\s*[\d๐-๙]+\s*ราชกิจจานุเบกษา[^\n]*'),
    re.compile(r'ฉบับพิเศษ\s*หน้า\s*[\d๐-๙]+[^\n]*ราชกิจจานุเบกษา[^\n]*'),
    re.compile(r'เล่ม\s*[\d๐-๙]+\s*หน้า\s*[\d๐-๙]+\s*ราชกิจจานุเบกษา[^\n]*'),
    re.compile(r'ตอนที่\s*[\d๐-๙]+\s*เล่ม\s*[\d๐-๙]+\s*ราชกิจจานุเบกษา[^\n]*'),
    re.compile(r'^ราชกิจจานุเบกษา[^\n]*$', re.MULTILINE),
    re.compile(r'^---+$', re.MULTILINE),
    re.compile(r'^\s*[\d๐-๙]+\s*$', re.MULTILINE),
    # inline ราชกิจจานุเบกษา ที่ OCR แทรกกลาง text
    re.compile(r'ฉบับพิเศษ\s*หน้า\s*\d+\s*ตอนที่\s*\d+\s*เล่ม\s*\d+\s*ราชกิจจานุเบกษา\s*\d+\s*\S+\s*\d+'),
    re.compile(r'กฤษณ์\s*\d+\s*เล่ม\s*\d+\s*ราชกิจจานุเบกษา[^\n]*'),
]

def _fix_sara_am(text: str) -> str:
    text = re.sub(r'([ก-ฮ])\s+า([ก-ฮ\s])', r'\1ำ\2', text)
    return text

def _thai_to_arabic(text: str) -> str:
    return text.translate(str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789'))

def clean_text(raw: str) -> str:
    text = raw
    for pat in _HEADER_PATTERNS:
        text = pat.sub(' ', text)
    text = _fix_sara_am(text)
    text = _thai_to_arabic(text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

## 2. STRUCTURE PARSING
---

In [9]:
# ── Boundary patterns ──────────────────────────────────────────
# หมวด / บท  (chapter level)
_RE_CHAPTER = re.compile(
    r'^(หมวด(?:ที่)?\s*\d+[^\n]*|บท(?:ที่)?\s*\d+[^\n]*)$',
    re.MULTILINE
)
_RE_SPECIAL_CHAPTER = re.compile(
    r'^(บททั่วไป|บทเฉพาะ(?:กาล)?|บทบัญญัติเฉพาะ(?:กาล)?|บทนำ|บทเบ็ดเตล็ด|บทสุดท้าย)\s*$',
    re.MULTILINE
)

# ส่วนที่  (part level — Bug 2 fix)
_RE_PART = re.compile(
    r'^(ส่วนที่\s*\d+[^\n]*)$',
    re.MULTILINE
)

# มาตรา  (section level)
_RE_SECTION_START = re.compile(r'มาตรา\s+(\d+)\s+', re.MULTILINE)

# ── Data classes ───────────────────────────────────────────────
@dataclass
class Section:
    section_number: int
    text: str

@dataclass
class Part:
    part_number: int
    part_title: str
    sections: list[Section] = field(default_factory=list)

@dataclass
class Chapter:
    chapter_number: int      # 0 = บททั่วไป, -1 = บทเฉพาะกาล/บทสุดท้าย
    chapter_title: str
    parts: list[Part] = field(default_factory=list)        # ส่วนที่ (อาจว่างถ้าไม่มี)
    sections: list[Section] = field(default_factory=list)  # มาตราที่ไม่อยู่ใน ส่วน


# ── Section parser (Bug 1 fix: position-based split) ──────────
def _parse_sections_from_segment(text: str) -> list[Section]:
    matches = list(_RE_SECTION_START.finditer(text))
    if not matches:
        return []

    raw_sections: list[Section] = []
    for i, m in enumerate(matches):
        sec_num = int(m.group(1))
        content_start = m.end()
        content_end   = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        raw_text = text[content_start:content_end]
        sec_text = re.sub(r'\s+', ' ', raw_text).strip()
        if sec_text:
            raw_sections.append(Section(section_number=sec_num, text=sec_text))

    # ── Bug 1b: merge duplicate section numbers ──────────────────
    merged: list[Section] = []
    for sec in raw_sections:
        if merged and merged[-1].section_number == sec.section_number:
            combined = merged[-1].text + ' ' + sec.text
            merged[-1] = Section(section_number=sec.section_number, text=combined)
        else:
            merged.append(sec)

    return merged


def _merge_cross_chapter_duplicates(chapters):
    seen = {}  # section_number -> chapter index
    for ci, ch in enumerate(chapters):
        new_secs = []
        for sec in ch.sections:
            n = sec.section_number
            if n in seen:
                orig_ci = seen[n]
                orig_secs = chapters[orig_ci].sections
                for si, s in enumerate(orig_secs):
                    if s.section_number == n:
                        orig_secs[si] = Section(n, s.text + ' ' + sec.text)
                        break
            else:
                seen[n] = ci
                new_secs.append(sec)
        ch.sections = new_secs
    return chapters


# ── Part parser (Bug 2 fix) ─────────────────────────────────────
def _parse_parts_from_segment(text: str) -> tuple[list[Part], list[Section]]:
    part_matches = list(_RE_PART.finditer(text))
    if not part_matches:
        return [], _parse_sections_from_segment(text)

    loose_sections = _parse_sections_from_segment(text[:part_matches[0].start()])
    parts = []

    for i, pm in enumerate(part_matches):
        part_end = part_matches[i + 1].start() if i + 1 < len(part_matches) else len(text)
        part_segment = text[pm.start():part_end]

        title_line = pm.group(1).strip()
        num_match = re.search(r'\d+', title_line)
        part_num = int(num_match.group()) if num_match else (i + 1)
        part_title = title_line

        sections = _parse_sections_from_segment(part_segment)
        parts.append(Part(part_number=part_num, part_title=part_title, sections=sections))

    return parts, loose_sections


# ── Chapter parser ──────────────────────────────────────────────
def _split_by_boundary(text: str) -> list[tuple[int, int, str]]:
    splits = []
    for m in _RE_SPECIAL_CHAPTER.finditer(text):
        title = m.group(1).strip()
        num = 0 if any(k in title for k in ['ทั่วไป', 'นำ']) else -1
        splits.append((m.start(), num, title))
    for m in _RE_CHAPTER.finditer(text):
        title = m.group(1).strip()
        num_m = re.search(r'\d+', title)
        if num_m:
            splits.append((m.start(), int(num_m.group()), title))
    splits.sort(key=lambda x: x[0])
    return splits


def _parse_full_text(full_text: str) -> tuple[str, list[Chapter]]:
    preamble = ""
    m1 = re.search(r'มาตรา\s+1\s+', full_text)
    if m1:
        preamble = re.sub(r'\s+', ' ', full_text[:m1.start()]).strip()[:2000]

    splits = _split_by_boundary(full_text)
    chapters: list[Chapter] = []

    if not splits:
        parts, loose = _parse_parts_from_segment(full_text)
        chapters.append(Chapter(chapter_number=0, chapter_title="(ไม่ระบุหมวด)",
                                parts=parts, sections=loose))
        return preamble, chapters

    for i, (pos, chap_num, chap_title) in enumerate(splits):
        end_pos = splits[i + 1][0] if i + 1 < len(splits) else len(full_text)
        segment = full_text[pos:end_pos]
        
        parts, loose_sections = _parse_parts_from_segment(segment)
        chapters.append(Chapter(
            chapter_number=chap_num,
            chapter_title=chap_title,
            parts=parts,
            sections=loose_sections,
        ))

    return preamble, chapters

## 3. CONSTITUTION TYPE DETECTION (Bug 3)
---

In [10]:
_AMENDMENT_KEYWORDS = [
    'แก้ไขเพิ่มเติม', 'ฉบับแก้ไข', 'แก้ไข พ.ศ.', 'amendment',
]

def _detect_constitution_type(raw_data: dict) -> tuple[str, Optional[int]]:
    name = raw_data.get('name_short', '') + ' ' + raw_data.get('full_text', '')[:500]
    is_amendment = any(kw in name for kw in _AMENDMENT_KEYWORDS)

    if is_amendment:
        year_match = re.search(r'แก้ไข.*?(\d{4})', name)
        amends_year = int(year_match.group(1)) if year_match else None
        return 'amendment', amends_year

    return 'original', None

## 4. MAIN PROCESSOR
---

In [11]:
def _chapter_total_sections(ch: Chapter) -> int:
    count = len(ch.sections)
    for p in ch.parts:
        count += len(p.sections)
    return count


def process_constitution_json(input_path: str, output_dir: str = ".") -> dict:
    input_path = Path(input_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*55}")
    print(f"Post-Processing v2: {input_path.name}")
    print(f"{'='*55}")

    with open(input_path, encoding='utf-8') as f:
        raw_data = json.load(f)

    year        = raw_data.get('year_th', 0)
    name        = raw_data.get('name_short', f'Constitution {year}')
    source_type = raw_data.get('source_type', 'unknown')
    pages       = raw_data.get('pages', [])

    if raw_data.get('full_text'):
        combined_raw = raw_data['full_text']
    else:
        combined_raw = "\n\n".join(
            p.get('raw_markdown', '') for p in pages if p.get('raw_markdown')
        )

    print(f"Cleaning ({len(combined_raw):,} chars)...")
    cleaned = clean_text(combined_raw)
    print(f"Clean: {len(cleaned):,} chars")

    print(f"Parsing structure (v2: หมวด→ส่วน→มาตรา)...")
    preamble, chapters = _parse_full_text(cleaned)

    total_sections = sum(_chapter_total_sections(c) for c in chapters)
    total_parts    = sum(len(c.parts) for c in chapters)
    print(f"{len(chapters)} หมวด | {total_parts} ส่วน | {total_sections} มาตรา")

    constitution_type, amends_year = _detect_constitution_type(raw_data)
    print(f"Type: {constitution_type}" +
          (f"(แก้ไขฉบับปี {amends_year})" if amends_year else ""))

    structured = {
        "id":                   raw_data.get('id', f'const_{year}'),
        "year_th":              year,
        "year_ce":              raw_data.get('year_ce', year - 543),
        "name_short":           name,
        "constitution_type":    constitution_type,
        "amends_year":          amends_year,
        "source_type":          source_type,
        "processing_method":    raw_data.get('processing_method', ''),
        "processed_at":         raw_data.get('processed_at', ''),
        "era":                  raw_data.get('era', ''),
        "regime_type":          raw_data.get('regime_type', ''),
        "preamble":             preamble,
        "summary": {
            "total_pages":    raw_data.get('total_pages', len(pages)),
            "total_chapters": len(chapters),
            "total_parts":    total_parts,
            "total_sections": total_sections,
            "total_chars":    len(cleaned),
        },
        "chapters": [
            {
                "chapter_number": c.chapter_number,
                "chapter_title":  c.chapter_title,
                "section_count":  _chapter_total_sections(c),
                "parts": [
                    {
                        "part_number":   p.part_number,
                        "part_title":    p.part_title,
                        "section_count": len(p.sections),
                        "sections": [
                            {"section_number": s.section_number, "text": s.text}
                            for s in p.sections
                        ],
                    }
                    for p in c.parts
                ],
                "sections": [
                    {"section_number": s.section_number, "text": s.text}
                    for s in c.sections
                ],
            }
            for c in chapters
        ],
    }

    json_out = output_dir / f"structured_{year}.json"
    with open(json_out, 'w', encoding='utf-8') as f:
        json.dump(structured, f, ensure_ascii=False, indent=2)
    print(f"JSON → {json_out}")

    csv_out = output_dir / f"sections_{year}.csv"
    _save_sections_csv(structured, csv_out)
    print(f"CSV  → {csv_out}")

    return structured


def _iter_all_sections(data: dict):
    base = {
        'constitution_id':   data['id'],
        'year_th':           data['year_th'],
        'year_ce':           data['year_ce'],
        'name_short':        data['name_short'],
        'constitution_type': data.get('constitution_type', 'original'),
        'amends_year':       data.get('amends_year', ''),
        'era':               data.get('era', ''),
        'regime_type':       data.get('regime_type', ''),
    }
    for chap in data['chapters']:
        chap_ctx = {
            **base,
            'chapter_number': chap['chapter_number'],
            'chapter_title':  chap['chapter_title'],
        }
        for sec in chap.get('sections', []):
            yield {**chap_ctx,
                   'part_number': '',
                   'part_title':  '',
                   'section_number': sec['section_number'],
                   'section_text':   sec['text']}
        for part in chap.get('parts', []):
            for sec in part.get('sections', []):
                yield {**chap_ctx,
                       'part_number': part['part_number'],
                       'part_title':  part['part_title'],
                       'section_number': sec['section_number'],
                       'section_text':   sec['text']}


def _save_sections_csv(data: dict, csv_path: Path):
    rows = list(_iter_all_sections(data))
    if not rows:
        return
    rows.sort(key=lambda r: r['section_number'])
    with open(csv_path, 'w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)


def batch_process(input_dir: str, output_dir: str = "structured_output"):
    input_dir = Path(input_dir)
    jsons = sorted(input_dir.glob("const_*.json")) or sorted(input_dir.glob("structured_*.json"))
    print(f"พบ {len(jsons)} ไฟล์")

    all_structured = []
    for j in jsons:
        try:
            all_structured.append(process_constitution_json(str(j), output_dir))
        except Exception as e:
            print(f" {j.name}: {e}")

    if all_structured:
        combined = Path(output_dir) / "all_sections_combined.csv"
        all_rows = []
        for data in all_structured:
            all_rows.extend(_iter_all_sections(data))
        all_rows.sort(key=lambda r: (r['year_th'], r['section_number']))
        with open(combined, 'w', encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=all_rows[0].keys())
            writer.writeheader()
            writer.writerows(all_rows)
        print(f"\n Combined CSV ({len(all_rows):,} rows) → {combined}")

    return all_structured

### Execution / CLI Support
---

In [17]:
# การรันใน Jupyter Notebook จะใช้แบบเรียก Function โดยตรงแทน argparse
INPUT_DIR = "./data" 
OUTPUT_DIR = "./struc-data-v2"

# สร้างโฟลเดอร์ปลายทางถ้ายังไม่มี
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# สั่งรันแบบ Batch (ประมวลผลทุกไฟล์ใน INPUT_DIR)
print(f"กำลังดึงข้อมูลจาก: {INPUT_DIR}")
batch_process(INPUT_DIR, OUTPUT_DIR)

def main():
    parser = argparse.ArgumentParser(description="Thai Constitution Post-Processor v2")
    parser.add_argument('--input',  required=True)
    parser.add_argument('--output', default='structured_output')
    parser.add_argument('--batch',  action='store_true')
    
    try:
        args = parser.parse_args()
        if args.batch:
            batch_process(args.input, args.output)
        else:
            process_constitution_json(args.input, args.output)
    except SystemExit:
        print("To run in Jupyter, please use process_constitution_json() or batch_process() directly.")

if __name__ == '__main__':
    # main()
    pass

กำลังดึงข้อมูลจาก: ./data
พบ 38 ไฟล์

Post-Processing v2: const_2475.json
Cleaning (18,946 chars)...
Clean: 17,654 chars
Parsing structure (v2: หมวด→ส่วน→มาตรา)...
9 หมวด | 0 ส่วน | 78 มาตรา
Type: original
JSON → struc-data-v2\structured_2475.json
CSV  → struc-data-v2\sections_2475.csv

Post-Processing v2: const_2482.json
Cleaning (988 chars)...
Clean: 856 chars
Parsing structure (v2: หมวด→ส่วน→มาตรา)...
1 หมวด | 0 ส่วน | 3 มาตรา
Type: amendment
JSON → struc-data-v2\structured_2482.json
CSV  → struc-data-v2\sections_2482.csv

Post-Processing v2: const_2483.json
Cleaning (1,506 chars)...
Clean: 1,343 chars
Parsing structure (v2: หมวด→ส่วน→มาตรา)...
1 หมวด | 0 ส่วน | 6 มาตรา
Type: amendment
JSON → struc-data-v2\structured_2483.json
CSV  → struc-data-v2\sections_2483.csv

Post-Processing v2: const_2485.json
Cleaning (1,743 chars)...
Clean: 1,670 chars
Parsing structure (v2: หมวด→ส่วน→มาตรา)...
1 หมวด | 0 ส่วน | 4 มาตรา
Type: amendment
JSON → struc-data-v2\structured_2485.json
CSV  → struc